In [8]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import os
import shutil
from sklearn.metrics import accuracy_score
%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

model_name = "MNIST_MLP_HGQ_StaticTraining_model"

In [9]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_test = X_test.astype("float32") / 255
y_test = keras.utils.to_categorical(y_test, 10)

In [10]:
from keras.models import load_model
import hgq.layers


model = load_model(f"{model_name}.keras")
score = model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9570 - loss: 0.4473


In [11]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense (QDense)                │ (None, 32)             │       100,485 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_1 (QDense)              │ (None, 32)             │         4,229 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense_2 (QDense)              │ (None, 10)             │         1,325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 265,089 (957.84 KB)

 Trainable params: 79,524 (310.64 KB)

 Non-trainable params: 26,515 (25.91 KB)

 Optimizer params: 159,050 (621.29 KB)

In [12]:
import hls4ml

output_dir = f"hls4ml_prj_{model_name}"
    
hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')

#hls_config['Model']['Strategy'] = 'Distributed Arithmetic'
#proj_name = f"{str(model_to_test)}_{str(model_revision)}_hls4ml_prj_{str(hls4ml_revision)}"

hls_model = hls4ml.converters.convert_from_keras_model(
    model,    
    backend='vitis',
    hls_config=hls_config,
    #io_type='io_stream',
    #proj_name = proj_name,
    output_dir=output_dir, 
    board       = 'kv260',
    part='xck26-sfvc784-2LV-c',
    clock_period='5',
)
hls_model.compile()

In [13]:
hls_model.build(csim=False, synth=True, cosim=False)


****** vitis-run v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Thu Apr 30 12:25:04 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
Sourcing Tcl script '/home/ncgadmin/DAT255/DAT255-project/MNIST/CNN_HGQ_StaticTraining/hls4ml_prj_MNIST_MLP_HGQ_StaticTraining_model/build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
Resolution: For help on HLS 200-2182 see docs.amd.com/access/sources/dita/topic?Doc_Version=2025.2%20English&url=ug1448-hls-guidance&resourceid=200-2182.html
INFO: [HLS 200-10] Creating and opening solution '/home/ncgadmin/DAT255/DAT255-project/MNIST/CNN_HGQ_StaticTraining/hls4ml_prj_MNIST_MLP_HGQ_StaticTraining_model/myproject_prj'.
INFO: [HLS 200-1510] Running: set_top myproject 
INFO: [HLS 200-1510] Running: add_files firmware/myproject.cpp -cflags -std=c++0x 
INFO: [HLS 200-10] 

Exception: Build failed for myproject. See logs for details.